# Alang 8ANI leakage-safe forecast study

Research cut: 12 August 2026. The production result is **NO CALL** at 5/10/15/30/45/60/90 calendar days. This notebook reruns the study and exposes the primary output tables.

## Validation contract

- Targets use the first Alang quote on/after horizon H, accepted only through H+4 calendar days.
- Every training label ends strictly before its forecast origin.
- Turkey and Bhavnagar use strict prior-date joins.
- Barchart roll-day returns are missing; momentum and volatility remain within one contract Symbol.
- Primary magnitude baseline is zero change. Direction retains down/flat/up.
- All models share identical OOS origins; dependence is addressed with non-overlap counts, phase cohorts and moving-block resampling.

In [ ]:
from pathlib import Path
import json, runpy
import pandas as pd

cwd = Path.cwd().resolve()
research = cwd if cwd.name == 'research' else (cwd / 'research' if (cwd / 'research').exists() else cwd.parent)
research

## Rerun all purged models

The script fits baseline, internal, local, Turkey, combined and pre-specified nonlinear models across all seven horizons, then rebuilds confidence intervals, stability diagnostics and max-null adjustments.

In [ ]:
runpy.run_path(str(research / 'src' / 'forecast_study_v2.py'), run_name='__main__')

In [ ]:
decisions = pd.read_csv(research / 'outputs' / 'v2' / 'horizon_decisions_v2.csv')
decision_cols = ['horizon','predicted_pct','predicted_price','lower80_price','upper80_price','prob_down','prob_flat','prob_up','oos_nonoverlap_n','holding_hurdle_rupee','decision','no_call_reasons']
decisions[decision_cols]

## Model backtests and the nearest complex-model candidates

In [ ]:
backtests = pd.read_csv(research / 'outputs' / 'v2' / 'backtest_summary_v2.csv')
view = ['model','horizon','raw_oos_n','nonoverlap_n','mae_skill_zero_log','mae_skill_zero_ci_low','mae_skill_zero_ci_high','brier_skill_frequency','accuracy','majority_accuracy','time_third_skill_min','phase_skill_min']
backtests.query("model != 'baseline'").sort_values('mae_skill_zero_log', ascending=False)[view].head(15)

## Pre-specified spread rules

This secondary study tests fixed ±1 trailing-z thresholds. It reports candidate-level familywise probabilities across 49 declared rule/horizon tests.

In [ ]:
rules = pd.read_csv(research / 'outputs' / 'rule_backtest_summary.csv')
rule_cols = ['rule','horizon','called_n','nonoverlap_n_median','raw_hit_rate_including_flat','economic_hit_rate','mean_signed_log_return','bootstrap_mean_signed_ci_low','bootstrap_mean_signed_ci_high','phase_mean_signed_min','max_shift_adjusted_p','passes_prespecified_rule_gate']
rules.sort_values('max_shift_adjusted_p')[rule_cols].head(15)

The plate–melt rule at 30 days is the nearest edge: adjusted p≈0.0227 and a positive block interval, but only 19 independent episodes. Because the gate requires N≥20, the result remains provisional.

## Physical-supply context

The AlangToday current-vessel snapshot is forward-only. The next cells show current inventory and illustrative release-kernel scenarios; they are not backtested forecasts.

In [ ]:
recent = pd.read_csv(research / 'outputs' / 'live_supply_recent_beachings.csv')
release = pd.read_csv(research / 'outputs' / 'live_supply_release_scenarios.csv')
display(recent)
release

## Reproducibility checks

In [ ]:
meta = json.loads((research / 'outputs' / 'v2' / 'study_metadata_v2.json').read_text(encoding='utf-8'))
{
    'assertions_passed': meta['assertions_passed'],
    'input_hashes_sha256': meta['input_hashes_sha256'],
    'target_raw_counts': meta['target_raw_counts'],
    'target_greedy_nonoverlap_counts': meta['target_greedy_nonoverlap_counts'],
    'common_oos_origin_counts': meta['common_oos_origin_counts'],
}